# What the published solutions add

The usual write-ups of this contest, read against the counts in this path, separate into moves that the manifest supports and moves that copy a label.

The supported moves are the ones the list already uses, with the definitions made explicit.

- Split `Cabin` into deck and side. Deck is almost a planet code: Earth on G, F, and E; Europa on A through E; Mars on D, E, and F.
- Trust `CryoSleep` first. It is 2483 of 3037 against 1789 of 5439.
- Treat a known positive bill during cryosleep as something this file does not contain. The count of such cells is 0.
- Keep `RoomService`, `Spa`, and `VRDeck` together, and keep `FoodCourt` and `ShoppingMall` out of that sum. The two pairs move in opposite directions.
- Treat age 0 through 12 as its own step: 564 of 806, and 429 of them are not already decided by cryosleep.
- Treat Europa on deck B or C as a step, and keep Maham in view: that step is 158 right and 52 wrong.
- Fill a blank `HomePlanet` from a groupmate only when the group has one known planet. That fills 90 of 201 blanks.

Applying that planet fill to the decision list changes 2 predictions. Both changes are corrections, so the training count moves from 6503 to 6505. Ninety filled cells, two extra correct fates.

The moves this path does not copy:

- Copying `Transported` inside a group. That is false on 797 of 1412 multi-person groups, and there is no train–test group to copy across.
- Encoding a cabin number by the fates of the other people who share it. That pastes the answer from one row onto a neighbor.
- A boosted tree over many crossed columns. Public write-ups of this contest commonly report a public score a little above 0.80 after that kind of model. This path does not fit one, and it does not claim that score. The audited held-out count here is 1331 of 1772.

A tree can represent the decision list, and it can also represent the thin starboard and Cancri experiments. The list is the version whose steps were counted one at a time.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Whole numbers are stored as decimals, such as 39.0.
def whole(text):
    return int(float(text))

def luxury(row):
    columns = ("RoomService", "Spa", "VRDeck")
    if any(row[column] == "" for column in columns):
        return None
    return sum(whole(row[column]) for column in columns)

def cryo_rule(row):
    return row["CryoSleep"] == "True"

def decision_list(row):
    # Step 1. Cryosleep.
    if row["CryoSleep"] == "True":
        return True
    # Step 2. Awake, and the luxury bill is at least 500.
    bill = luxury(row)
    if row["CryoSleep"] == "False" and bill is not None and bill >= 500:
        return False
    # Step 3. Age at most 12.
    if row["Age"] != "" and whole(row["Age"]) <= 12:
        return True
    # Step 4. Europa, deck B or deck C.
    if row["Cabin"] != "":
        deck = row["Cabin"].split("/")[0]
        if row["HomePlanet"] == "Europa" and deck in ("B", "C"):
            return True
    # Step 5. Everyone else stays.
    return False

print("rules ready", len(train))


rules ready 8693


In [4]:
# Filling a blank planet from a unique group planet changes two predictions.
from collections import defaultdict
groups = defaultdict(list)
for index, row in enumerate(train):
    groups[row["PassengerId"].split("_")[0]].append(index)

def planet_of(index):
    row = train[index]
    if row["HomePlanet"] != "":
        return row["HomePlanet"]
    known = {
        train[other]["HomePlanet"]
        for other in groups[row["PassengerId"].split("_")[0]]
        if train[other]["HomePlanet"] != ""
    }
    if len(known) == 1:
        return next(iter(known))
    return ""

def predict(row, planet):
    if row["CryoSleep"] == "True":
        return True
    bill = luxury(row)
    if row["CryoSleep"] == "False" and bill is not None and bill >= 500:
        return False
    if row["Age"] != "" and whole(row["Age"]) <= 12:
        return True
    if row["Cabin"] != "":
        deck = row["Cabin"].split("/")[0]
        if planet == "Europa" and deck in ("B", "C"):
            return True
    return False

base = filled = changed = 0
for index, row in enumerate(train):
    fate = row["Transported"] == "True"
    before = predict(row, row["HomePlanet"])
    after = predict(row, planet_of(index))
    base += before == fate
    filled += after == fate
    changed += before != after
print(base, filled, changed)
assert (base, filled, changed) == (6503, 6505, 2)


6503 6505 2


## Pitfall

A published score near 0.80 is someone else's leaderboard result. It is not a count you can recompute from `train.csv` alone, because the contest grades `test.csv` on fates that are not in the file. Reporting 1331 of 1772 says which passengers were scored and which rule was used.

## Summary

The group planet fill is worth two training rows, 6503 to 6505. Group fates are mixed on 797 groups. The number this path stands behind is the held-out 1331 of 1772, not a leaderboard entry.
